In [ ]:
import os
import subprocess
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import random
import re

# --- CẤU HÌNH ---
FILENAME = '../../data/raw/homedy_raw.csv' # Tên file CSV của bạn
BASE_URL = "https://homedy.com/cho-thue-nha-tro-phong-tro-tp-ho-chi-minh"
START_PAGE = 41
END_PAGE = 104

# --- KHỞI TẠO DRIVER ---
def get_driver():
    try:
        subprocess.run(['pkill', '-9', 'chrome'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except: pass
    
    options = Options()
    options.add_argument('--headless=new')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_page_load_timeout(60)
    return driver

# --- CÁC HÀM XỬ LÝ ---
def clean_price(text):
    if not text: return 0
    try:
        text = text.lower().replace(',', '.')
        nums = re.findall(r"[-+]?\d*\.\d+|\d+", text)
        if not nums: return 0
        val = float(nums[0])
        if 'triệu' in text: return int(val * 1000000)
        if 'tỷ' in text: return int(val * 1000000000)
        if 'nghìn' in text: return int(val * 1000)
        return int(val)
    except: return 0

def clean_area(text):
    if not text: return 0.0
    try:
        text = text.lower().replace(',', '.')
        nums = re.findall(r"[-+]?\d*\.\d+|\d+", text)
        return float(nums[0]) if nums else 0.0
    except: return 0.0

def clean_district(text):
    if not text or text == "N/A": return "N/A"
    parts = text.split(',')
    if len(parts) >= 2:
        return parts[-2].strip()
    return text.strip()

def get_text(element, selector):
    try:
        return element.find_element(By.CSS_SELECTOR, selector).text.strip()
    except:
        return "N/A"

def get_full_description(driver, link):
    """Vào trang chi tiết lấy mô tả full"""
    try:
        driver.get(link)
        try:
            WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
            # Các class có thể chứa mô tả
            selectors = [".description", ".product-description", "#description-content", ".description-content"]
            for sel in selectors:
                try:
                    elem = driver.find_element(By.CSS_SELECTOR, sel)
                    if elem.is_displayed():
                        return elem.text.strip().replace('\n', ' ')
                except: continue
        except: pass
        return "N/A"
    except: return "Error"

# --- CHẠY CHÍNH ---
def run_scraper():
    driver = None
    current_index = 0
    batch_data = [] # Bộ nhớ tạm chứa dữ liệu của 1 trang
    
    # Kiểm tra file cũ
    if os.path.exists(FILENAME):
        try:
            df_check = pd.read_csv(FILENAME)
            if not df_check.empty:
                current_index = df_check['index'].max() + 1
                print(f"📂 Tim thay file cu '{FILENAME}'. Tiep tuc tu Index: {current_index}")
        except: pass

    try:
        driver = get_driver()
        print(f"🚀 Bat dau cao du lieu tu trang {START_PAGE} den {END_PAGE}...")

        for page in range(START_PAGE, END_PAGE + 1):
            url = f"{BASE_URL}/p{page}"
            print(f"\n--- DANG XU LY TRANG {page} ---")
            
            try:
                driver.get(url)
                WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.CSS_SELECTOR, ".product-item")))
            except:
                print("   ! Loi tai trang hoac het tin.")
                continue

            # 1. Lấy thông tin cơ bản từ list
            items = driver.find_elements(By.CSS_SELECTOR, ".product-item")
            print(f"   -> Tim thay {len(items)} tin. Dang lay Link & Info co ban...")
            
            temp_list = []
            for item in items:
                try:
                    # Selector chuẩn cho Homedy: h3 a
                    try:
                        title_el = item.find_element(By.CSS_SELECTOR, "h3 a")
                    except:
                        title_el = item.find_element(By.CSS_SELECTOR, ".title a")
                        
                    title = title_el.text.strip()
                    link = title_el.get_attribute('href')
                    if link and not link.startswith('http'):
                        link = "https://homedy.com" + link
                    
                    price_raw = get_text(item, ".price")
                    area_raw = get_text(item, ".acreage")
                    district_raw = get_text(item, ".address")
                    
                    temp_list.append({
                        "link": link,
                        "title": title,
                        "price_raw": price_raw,
                        "area_raw": area_raw,
                        "district_raw": district_raw
                    })
                except: continue
            
            # 2. Vào chi tiết từng tin để lấy Full Desc
            batch_data = [] # Reset mỗi khi sang trang mới
            print(f"   -> Bat dau vao chi tiet {len(temp_list)} tin...")
            
            for i, temp in enumerate(temp_list):
                full_desc = get_full_description(driver, temp['link'])
                
                row = {
                    'index': current_index,
                    'url': temp['link'],
                    'title': temp['title'],
                    'price': clean_price(temp['price_raw']),
                    'area': clean_area(temp['area_raw']),
                    'district': clean_district(temp['district_raw']),
                    'description': full_desc, # FULL TEXT
                    'source': 'homedy.com'
                }
                batch_data.append(row)
                current_index += 1
                
                print(f"     + [{i+1}/{len(temp_list)}] {temp['title'][:30]}... | Desc: {len(full_desc)} chars")
                time.sleep(random.uniform(1.5, 2.5)) # Nghỉ ngắn

            # 3. LOGIC LƯU FILE (LƯU NGAY SAU MỖI TRANG)
            if batch_data:
                df = pd.DataFrame(batch_data)
                
                # Nếu file chưa tồn tại hoặc rỗng thì ghi header
                hdr = not os.path.exists(FILENAME) or os.path.getsize(FILENAME) == 0
                
                df.to_csv(FILENAME, mode='a', header=hdr, index=False, encoding='utf-8-sig')
                print(f"\n💾 ✅ Đã thêm {len(batch_data)} tin mới vào file: {FILENAME}")
            else:
                print(f"\n⚠️ Trang {page} khong co du lieu de luu.")
            
            time.sleep(3)

    except Exception as e:
        print(f"❌ LOI CHUONG TRINH: {e}")
        if batch_data:
            print("💾 Dang luu du lieu con lai truoc khi thoat...")
            df = pd.DataFrame(batch_data)
            df.to_csv(FILENAME, mode='a', header=False, index=False, encoding='utf-8-sig')

    finally:
        if driver: driver.quit()
        print("\n🎉 CHUONG TRINH KET THUC!")

if __name__ == "__main__":
    run_scraper()